# Custos de produção de milho (2ª safra) — Paraná

Este notebook lê a planilha `milho_2a_safra_serie_historica_2005-2025.xls` (série histórica CONAB
de custos de produção) e monta um conjunto de DataFrames com os **custos de produção do milho por
cidade do Paraná**, permitindo consultar qualquer intervalo de anos.

## O que a planilha contém

- 254 abas no total. Cada aba representa **uma combinação cidade-UF-ano** (ex.: `Londrina-PR-2020`),
  exceto a primeira aba (`Índice`).
- No Paraná (`-PR-`) existem **85 abas**, cobrindo 6 cidades:
  `Assis Chateaubriand`, `Campo Mourão`, `Francisco Beltrão`, `Londrina`, `M. Cândido Rondon`, `Ubiratã`
  — cada uma com anos diferentes disponíveis (a série mais longa é Londrina e Campo Mourão, de 1999/2002 a 2025).
- Cada aba segue o layout padrão CONAB de "Custo de Produção", mas o **formato da planilha mudou ao
  longo do tempo** (o relatório é gerado por um sistema diferente conforme o ano):
  - Anos mais antigos: layout simples de 4 colunas (`item`, `R$/ha`, `R$/60kg`, `%`).
  - Anos intermediários: layout de 13 colunas com células mescladas (o valor de "custo por ha" das
    linhas de item aparece deslocado em relação ao cabeçalho, por causa do merge de células no Excel).
  - Anos mais recentes (2025): layout limpo de 5 colunas (`item`, `R$/ha`, `R$/60kg`, `%CV`, `%CT`).

  Por isso, em vez de ler por índice fixo de coluna, o parser abaixo localiza o cabeçalho
  ("DISCRIMINAÇÃO") em cada aba e usa a **ordem** dos valores numéricos de cada linha (da esquerda
  para a direita) para inferir `custo_rs_ha`, `custo_rs_60kg`, `participacao_pct` e
  `participacao_ct_pct` — isso funciona nos três formatos sem precisar tratar cada um manualmente.

## Estrutura de dados proposta

Como você sugeriu, cada aba do Paraná vira um dicionário aninhado:

```python
custos_pr[cidade][ano] -> DataFrame com os itens de custo daquela cidade/ano
```

Além disso, monto um único DataFrame "longo" (`df_pr_long`, uma linha por item de custo por
cidade/ano) porque é o formato mais fácil de **filtrar por intervalo de anos e comparar cidades**
lado a lado — o dicionário fica ótimo para inspecionar uma cidade/ano específico, e o DataFrame longo
fica ótimo para consultas e pivôs. No fim, uma função `consultar_custos(...)` e um pequeno filtro
interativo (ipywidgets) permitem escolher cidades + intervalo de anos e ver o resultado direto como
DataFrame.

In [1]:
import re
import pandas as pd
import numpy as np

ARQUIVO = "milho_2a_safra_serie_historica_2005-2025.xls"

xls = pd.ExcelFile(ARQUIVO)
print(f"Total de abas no arquivo: {len(xls.sheet_names)}")

Total de abas no arquivo: 254


## 1. Localizar as abas do Paraná

Os nomes das abas seguem o padrão `Cidade-UF-Ano`. Filtramos as que terminam em `-PR-<ano>` e
extraímos cidade e ano com regex.

In [2]:
PADRAO_ABA_PR = re.compile(r"^(.*)-PR-(\d{4})$")

abas_pr = []
for nome in xls.sheet_names:
    m = PADRAO_ABA_PR.match(nome)
    if m:
        cidade, ano = m.group(1), int(m.group(2))
        abas_pr.append({"aba": nome, "cidade": cidade, "ano": ano})

abas_pr_df = pd.DataFrame(abas_pr).sort_values(["cidade", "ano"]).reset_index(drop=True)
print(f"Abas do Paraná encontradas: {len(abas_pr_df)}")
print(f"Cidades: {sorted(abas_pr_df['cidade'].unique())}")
abas_pr_df.groupby("cidade")["ano"].agg(["min", "max", "count"])

Abas do Paraná encontradas: 85
Cidades: ['Assis Chateaubriand', 'Campo Mourão', 'Francisco Beltrão', 'Londrina', 'M. Cândido Rondon', 'Ubiratã']


,min,max,count
cidade,,,
Assis Chateaubriand,2019,2023,5
Campo Mourão,1999,2025,25
Francisco Beltrão,2019,2025,7
Londrina,1999,2025,27
M. Cândido Rondon,2023,2025,3
Ubiratã,2008,2025,18


## 2. Parser genérico de uma aba de custo

Independente do formato (4, 5 ou 13 colunas), a função:

1. Encontra a linha e a coluna do cabeçalho `DISCRIMINAÇÃO`.
2. Encontra a linha de rodapé (`ELABORAÇÃO: ...`) para saber onde a tabela termina.
3. Extrai a produtividade média (kg/ha) informada no topo da aba.
4. Para cada linha de item, pega o rótulo (texto na coluna do cabeçalho) e os valores numéricos
   presentes à direita dele, **na ordem em que aparecem** — o primeiro valor numérico é sempre o
   custo em R$/ha, o segundo o custo em R$/60kg, e os seguintes são percentuais de participação.
5. Linhas sem nenhum valor numérico são títulos de seção (ex. `I - DESPESAS DE CUSTEIO`) e viram a
   "seção" associada aos itens seguintes, em vez de virar uma linha de custo.

In [4]:
def parse_aba_custo(xls, aba):
    '''Retorna (produtividade_kg_ha, DataFrame de itens de custo) para uma aba do arquivo CONAB.'''
    df = xls.parse(aba, header=None)
    n_linhas, n_colunas = df.shape

    linha_cab, col_rotulo = None, None
    for r in range(n_linhas):
        for c in range(n_colunas):
            v = df.iat[r, c]
            if isinstance(v, str) and "DISCRIMINA" in v.upper():
                linha_cab, col_rotulo = r, c
                break
        if linha_cab is not None:
            break
    if linha_cab is None:
        raise ValueError(f"Cabeçalho 'DISCRIMINAÇÃO' não encontrado na aba {aba!r}")

    linha_fim = n_linhas
    for r in range(linha_cab + 1, n_linhas):
        if any(isinstance(df.iat[r, c], str) and "ELABORA" in df.iat[r, c].upper() for c in range(n_colunas)):
            linha_fim = r
            break

    produtividade = None
    for r in range(0, linha_cab):
        for c in range(n_colunas):
            v = df.iat[r, c]
            if isinstance(v, str) and "PRODUTIVID" in v.upper():
                numeros = []
                for c2 in range(n_colunas):
                    v2 = df.iat[r, c2]
                    if isinstance(v2, (int, float)) and pd.notna(v2):
                        numeros.append(v2)
                    elif isinstance(v2, str):
                        m = re.search(r"([\d.,]+)\s*kg", v2, re.IGNORECASE)
                        if m:
                            numeros.append(float(m.group(1).replace(".", "").replace(",", ".")))
                if numeros:
                    produtividade = numeros[0]

    secao = None
    registros = []
    for r in range(linha_cab + 1, linha_fim):
        rotulo = df.iat[r, col_rotulo]
        if not isinstance(rotulo, str) or not rotulo.strip():
            continue
        rotulo = rotulo.strip()

        valores = []
        for c in range(col_rotulo + 1, n_colunas):
            v = df.iat[r, c]
            if isinstance(v, (int, float)) and pd.notna(v):
                valores.append(v)

        if not valores:
            secao = rotulo
            continue

        registro = {"secao": secao, "item": rotulo, "custo_rs_ha": valores[0]}
        if len(valores) >= 2:
            registro["custo_rs_60kg"] = valores[1]
        if len(valores) >= 3:
            registro["participacao_pct"] = valores[2]
        if len(valores) >= 4:
            registro["participacao_ct_pct"] = valores[3]
        registros.append(registro)

    return produtividade, pd.DataFrame(registros)


# teste rápido em uma aba de cada formato
for aba_teste in ["Londrina-PR-1999", "Londrina-PR-2020", "Londrina-PR-2025"]:
    prod, d = parse_aba_custo(xls, aba_teste)
    print(f"{aba_teste}: produtividade={prod} kg/ha, {len(d)} itens de custo")

Londrina-PR-1999: produtividade=3500 kg/ha, 31 itens de custo
Londrina-PR-2020: produtividade=5700.0 kg/ha, 46 itens de custo
Londrina-PR-2025: produtividade=5700.0 kg/ha, 46 itens de custo


## 3. Montar o dicionário `custos_pr[cidade][ano]` e o DataFrame longo `df_pr_long`

In [5]:
custos_pr = {}
registros_long = []

for _, linha in abas_pr_df.iterrows():
    aba, cidade, ano = linha["aba"], linha["cidade"], linha["ano"]
    produtividade, df_itens = parse_aba_custo(xls, aba)

    custos_pr.setdefault(cidade, {})[ano] = df_itens

    df_itens_long = df_itens.copy()
    df_itens_long.insert(0, "cidade", cidade)
    df_itens_long.insert(1, "uf", "PR")
    df_itens_long.insert(2, "ano", ano)
    df_itens_long["produtividade_kg_ha"] = produtividade
    registros_long.append(df_itens_long)

df_pr_long = pd.concat(registros_long, ignore_index=True)
print(f"df_pr_long: {df_pr_long.shape[0]} linhas de custo, {df_pr_long['cidade'].nunique()} cidades, "
      f"anos de {df_pr_long['ano'].min()} a {df_pr_long['ano'].max()}")
df_pr_long.head(10)

df_pr_long: 3816 linhas de custo, 6 cidades, anos de 1999 a 2025


C:\Users\fabri\AppData\Local\Temp\ipykernel_31580\4006470311.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_pr_long = pd.concat(registros_long, ignore_index=True)


,cidade,uf,ano,secao,item,custo_rs_ha,custo_rs_60kg,participacao_pct,produtividade_kg_ha,participacao_ct_pct
0,Assis Chateaubriand,PR,2019,I - DESPESAS DE CUSTEIO DA LAVOURA,1 - Operação com animal,0.00,0.00,0.000000,6000.0,NaN
1,Assis Chateaubriand,PR,2019,I - DESPESAS DE CUSTEIO DA LAVOURA,2 - Operação com avião,0.00,0.00,0.000000,6000.0,NaN
2,Assis Chateaubriand,PR,2019,3 - Operação com máquinas próprias:,3.1 - Tratores e Colheitadeiras,104.28,1.03,0.030501,6000.0,NaN
3,Assis Chateaubriand,PR,2019,3 - Operação com máquinas próprias:,3.2 - Conjunto de Irrigação,0.00,0.00,0.000000,6000.0,NaN
4,Assis Chateaubriand,PR,2019,3 - Operação com máquinas próprias:,4 - Aluguel de máquinas,146.74,1.47,0.042921,6000.0,NaN
5,Assis Chateaubriand,PR,2019,3 - Operação com máquinas próprias:,5 - Aluguel de animais,0.00,0.00,0.000000,6000.0,NaN
6,Assis Chateaubriand,PR,2019,3 - Operação com máquinas próprias:,6 - Mão de obra,0.00,0.00,0.000000,6000.0,NaN
7,Assis Chateaubriand,PR,2019,3 - Operação com máquinas próprias:,7 - Administrador Rural,69.88,0.68,0.020440,6000.0,NaN
8,Assis Chateaubriand,PR,2019,3 - Operação com máquinas próprias:,8 - Sementes,750.00,7.50,0.219371,6000.0,NaN
9,Assis Chateaubriand,PR,2019,3 - Operação com máquinas próprias:,9 - Fertilizantes,720.00,7.20,0.210596,6000.0,NaN


In [6]:
# Exemplo de acesso via dicionário, como você propôs:
custos_pr["Londrina"][2023]

,secao,item,custo_rs_ha,custo_rs_60kg,participacao_pct,participacao_ct_pct
0,I - DESPESAS DO CUSTEIO,1 - Operação com animal,0.00,0.00000,0.00,0.00
1,I - DESPESAS DO CUSTEIO,2 - Operação com Avião,0.00,0.00000,0.00,0.00
2,3 - Operação com máquinas:,3.1 - Tratores e Colheitadeiras,577.35,6.07735,11.08,6.76
3,3 - Operação com máquinas:,3.2 - Conjunto de Irrigação,0.00,0.00000,0.00,0.00
4,3 - Operação com máquinas:,4 - Aluguel de Máquinas,0.00,0.00000,0.00,0.00
5,3 - Operação com máquinas:,5 - Aluguel de Animais,0.00,0.00000,0.00,0.00
6,3 - Operação com máquinas:,6 - Mão de obra,13.10,0.13789,0.25,0.15
7,3 - Operação com máquinas:,7 - Administrador,339.48,3.57348,6.52,3.97
8,3 - Operação com máquinas:,8 - Sementes e mudas,1400.00,14.73684,26.87,16.39
9,3 - Operação com máquinas:,9 - Fertilizantes,1210.02,12.73705,23.23,14.16


## 4. Consulta por intervalo de tempo

`consultar_custos` filtra `df_pr_long` por cidade(s), intervalo de anos e (opcionalmente) por item(ns)
de custo, devolvendo uma tabela pivotada: uma linha por item de custo, uma coluna por
cidade/ano, valores em R$/ha.

In [7]:
def consultar_custos(cidades=None, ano_inicio=None, ano_fim=None, itens=None, metrica="custo_rs_ha"):
    '''
    cidades: lista de cidades do PR (None = todas)
    ano_inicio, ano_fim: intervalo de anos, inclusive (None = sem limite)
    itens: lista de itens de custo a filtrar (None = todos os itens/linhas da planilha)
    metrica: 'custo_rs_ha', 'custo_rs_60kg', 'participacao_pct' ou 'participacao_ct_pct'
    '''
    dados = df_pr_long.copy()

    if cidades is not None:
        dados = dados[dados["cidade"].isin(cidades)]
    if ano_inicio is not None:
        dados = dados[dados["ano"] >= ano_inicio]
    if ano_fim is not None:
        dados = dados[dados["ano"] <= ano_fim]
    if itens is not None:
        dados = dados[dados["item"].isin(itens)]

    tabela = dados.pivot_table(
        index="item",
        columns=["cidade", "ano"],
        values=metrica,
        aggfunc="first",
    )
    return tabela.sort_index(axis=1)


# Exemplo: custo por ha (R$/ha) de todas as cidades do PR entre 2019 e 2023
consultar_custos(ano_inicio=2019, ano_fim=2023)

cidade                           Assis Chateaubriand                           \
ano                                             2019    2020    2021     2022   
item                                                                            
1 - Operação com animal                      0.00000    0.00    0.00     0.00   
10 - Agrotóxicos                           410.20000  414.45  504.30   744.44   
11 - Receita                                     NaN    0.00    0.00     0.00   
11 - Água                                    0.00000     NaN     NaN      NaN   
12 - Receita                                 0.00000     NaN     NaN      NaN   
...                                              ...     ...     ...      ...   
TOTAL DAS OUTRAS DESPESAS (B)              357.85000  363.84  546.10   778.49   
TOTAL DE DEPRECIAÇÕES (E)                  117.15000  112.59  184.65   294.52   
TOTAL DE OUTROS CUSTOS FIXOS (F)            50.28547  159.87  220.72   307.94   
TOTAL DE RENDA DE FATORES (F)                    NaN     NaN  341.09  1112.19   
TOTAL DE RENDA DE FATORES (I)                    NaN  668.54     NaN      NaN   

cidade                                    Campo Mourão                   \
ano                                  2023         2019     2020    2021   
item                                                                      
1 - Operação com animal              0.00         0.00     0.00    0.00   
10 - Agrotóxicos                   632.60       574.85   565.22  636.88   
11 - Receita                         0.00          NaN     0.00    0.00   
11 - Água                             NaN         0.00      NaN     NaN   
12 - Receita                          NaN         0.00      NaN     NaN   
...                                   ...          ...      ...     ...   
TOTAL DAS OUTRAS DESPESAS (B)      686.85       588.47   656.01  789.18   
TOTAL DE DEPRECIAÇÕES (E)          312.20        86.22    98.35  119.68   
TOTAL DE OUTROS CUSTOS FIXOS (F)   304.78       178.69   233.93  319.95   
TOTAL DE RENDA DE FATORES (F)     3043.88          NaN      NaN  339.65   
TOTAL DE RENDA DE FATORES (I)         NaN       818.35  1101.15     NaN   

cidade                                              ... Londrina          \
ano                                  2022     2023  ...     2020    2021   
item                                                ...                    
1 - Operação com animal              0.00     0.00  ...     0.00    0.00   
10 - Agrotóxicos                   925.72  1039.20  ...   371.23  477.76   
11 - Receita                         0.00     0.00  ...     0.00    0.00   
11 - Água                             NaN      NaN  ...      NaN     NaN   
12 - Receita                          NaN      NaN  ...      NaN     NaN   
...                                   ...      ...  ...      ...     ...   
TOTAL DAS OUTRAS DESPESAS (B)     1051.11   913.42  ...   401.27  494.74   
TOTAL DE DEPRECIAÇÕES (E)          257.76   308.85  ...   290.99  382.59   
TOTAL DE OUTROS CUSTOS FIXOS (F)   364.31   703.49  ...   552.06  827.93   
TOTAL DE RENDA DE FATORES (F)     1259.81  2694.92  ...      NaN  216.80   
TOTAL DE RENDA DE FATORES (I)         NaN      NaN  ...   741.74     NaN   

cidade                                            M. Cândido Rondon Ubiratã  \
ano                                 2022     2023              2023    2019   
item                                                                          
1 - Operação com animal             0.00     0.00           0.00000    0.00   
10 - Agrotóxicos                  910.62   808.96         982.24000  384.40   
11 - Receita                        0.00     0.00               NaN     NaN   
11 - Água                            NaN      NaN           0.00000    0.00   
12 - Receita                         NaN      NaN           0.00000    0.00   
...                                  ...      ...               ...     ...   
TOTAL DAS OUTRAS DESPESAS (B)     730.

In [8]:
# Exemplo: evolução do item "CUSTO TOTAL (H+I=J)" em Londrina e Campo Mourão, últimos 10 anos
consultar_custos(
    cidades=["Londrina", "Campo Mourão"],
    ano_inicio=2015,
    ano_fim=2025,
    itens=["CUSTO TOTAL (H+I=J)"],
)

cidade              Campo Mourão                                               \
ano                         2015     2016     2017     2018     2019     2020   
item                                                                            
CUSTO TOTAL (H+I=J)      2758.15  2922.61  2936.79  2863.31  3815.39  4301.78   

cidade                                                  ... Londrina           \
ano                     2021     2022     2023    2024  ...     2016     2017   
item                                                    ...                     
CUSTO TOTAL (H+I=J)  4468.71  7378.56  8555.43  7527.7  ...  2810.18  2870.23   

cidade                                                                     \
ano                     2018     2019     2020     2021     2022     2023   
item                                                                        
CUSTO TOTAL (H+I=J)  3602.93  3604.59  4445.11  4986.62  8119.33  8542.61   

cidade                                 
ano                     2024     2025  
item                                   
CUSTO TOTAL (H+I=J)  7428.42  7498.76  

[1 rows x 22 columns]

## 5. Filtro interativo (widgets)

Selecione cidade(s) e o intervalo de anos abaixo — a tabela é atualizada automaticamente.

In [9]:
import ipywidgets as widgets
from IPython.display import display

cidades_disponiveis = sorted(df_pr_long["cidade"].unique())
ano_min, ano_max = int(df_pr_long["ano"].min()), int(df_pr_long["ano"].max())

sel_cidades = widgets.SelectMultiple(
    options=cidades_disponiveis,
    value=tuple(cidades_disponiveis),
    description="Cidades:",
    rows=6,
)
sel_anos = widgets.IntRangeSlider(
    value=[ano_max - 5, ano_max],
    min=ano_min,
    max=ano_max,
    step=1,
    description="Anos:",
    continuous_update=False,
)
sel_item = widgets.Dropdown(
    options=["(todos os itens)"] + sorted(df_pr_long["item"].unique()),
    value="CUSTO TOTAL (H+I=J)" if "CUSTO TOTAL (H+I=J)" in df_pr_long["item"].unique() else "(todos os itens)",
    description="Item:",
)
saida = widgets.Output()


def atualizar(*_):
    saida.clear_output()
    itens_filtro = None if sel_item.value == "(todos os itens)" else [sel_item.value]
    tabela = consultar_custos(
        cidades=list(sel_cidades.value),
        ano_inicio=sel_anos.value[0],
        ano_fim=sel_anos.value[1],
        itens=itens_filtro,
    )
    with saida:
        display(tabela)


for w in (sel_cidades, sel_anos, sel_item):
    w.observe(atualizar, names="value")

atualizar()
display(widgets.VBox([sel_cidades, sel_anos, sel_item, saida]))

In [10]:
df_pr_long.to_csv("df_pr_long.csv", index=False, encoding="utf-8-sig")
